# LF2 — nhãn hữu dụng cho tác vụ bệnh trên lá (classifier ảnh, cross-fitting)

Ảnh hữu dụng cho tác vụ bệnh lá nếu model bệnh-lá hạ nguồn phân đúng lớp foliar. Xem thêm tại `docs/LF2_Methodology.md`.

Bộ bệnh (Mendeley `gh56wbsnj5`), 2 lớp foliar: Gray Leaf Spot (2135 ảnh), Leaf Rot (1673 ảnh). Ảnh không có bản augment ×3.

## Cài đặt

In [1]:
%pip install -q torchvision

Note: you may need to restart the kernel to use updated packages.


## Cấu hình

In [2]:
import platform
import hashlib
import random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torchvision

RUNNER    = 'local'        # 'local' | 'kaggle'
SEED      = 42
K         = 5              # so fold cross-fitting
CONF_TAU  = 0.60           # confidence < CONF_TAU -> abstain
ARCH      = 'mobilenet_v3_small'
IMGSZ     = 224
EPOCHS    = 30
BATCH     = 32
LR        = 0.0005
PATIENCE  = 5              # early-stopping: dung neu val loss khong cai thien sau PATIENCE epoch
VAL_FRAC  = 0.15           # ti le val noi bo (tach tu K-1 fold train) cho early-stopping

TASK     = '2_foliar_disease'
CLASSES  = ['Gray Leaf Spot', 'Leaf Rot']
SOURCE   = 'coconut-tree-disease'
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cpu'
CUDA_NAME = 'none'
if torch.cuda.is_available():
    DEVICE = 'cuda'
    CUDA_NAME = torch.cuda.get_device_name(0)
elif torch.backends.mps.is_available():
    DEVICE = 'mps'

print('python:', platform.python_version())
print('torch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('device:', DEVICE, '| cuda:', CUDA_NAME)
print('runner:', RUNNER)
print('seed:', SEED, '| K:', K, '| conf_tau:', CONF_TAU)
print('arch:', ARCH, '| imgsz:', IMGSZ, '| epochs:', EPOCHS, '| batch:', BATCH, '| lr:', LR)
print('patience:', PATIENCE, '| val_frac:', VAL_FRAC)

python: 3.10.13
torch: 2.12.0
torchvision: 0.27.0
device: mps | cuda: none
runner: local
seed: 42 | K: 5 | conf_tau: 0.6
arch: mobilenet_v3_small | imgsz: 224 | epochs: 30 | batch: 32 | lr: 0.0005
patience: 5 | val_frac: 0.15


In [3]:
def build_root(runner):
    if runner == 'local':
        for cand in (Path.cwd(), Path.cwd().parent):
            if (cand/'Dataset'/'Coconut Tree Disease Dataset').exists():
                return cand
        raise SystemExit('Khong thay Dataset/Coconut Tree Disease Dataset — chay notebook ben trong repo coconut-iqa, kiem tra lai duong dan')
    if runner == 'kaggle':
        root = Path('/kaggle/input/coconut-iqa')
        if not (root/'Dataset'/'Coconut Tree Disease Dataset').exists():
            raise SystemExit('Khong thay /kaggle/input/coconut-iqa/Dataset/Coconut Tree Disease Dataset — kiem tra lai ten dataset da add')
        return root
    raise SystemExit("RUNNER phai la 'local' hoac 'kaggle'")

ROOT      = build_root(RUNNER)
BASE      = ROOT/'Dataset'/'Coconut Tree Disease Dataset'
VOTES_OUT = ROOT/'labels'/'votes'/'lf2_foliar.csv'
OUT_DIR   = ROOT/'labels'/'lf2_foliar'
RUN_DIR   = OUT_DIR/'runs'
OUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)

# Nap module dung chung (schema phieu + writer). Khong import duoc .ipynb -> dung %run.
UTILS = ROOT/'src'/'utils'/'lf_io.ipynb'
if not UTILS.exists():
    raise SystemExit('Khong thay ' + str(UTILS))
get_ipython().run_line_magic('run', str(UTILS))

print('ROOT:', ROOT)
print('VOTES_OUT:', VOTES_OUT)

ROOT: /Users/peggy/Documents/Projects/HK2/coconut-iqa
VOTES_OUT: /Users/peggy/Documents/Projects/HK2/coconut-iqa/labels/votes/lf2_foliar.csv


/Users/peggy/.pyenv/versions/3.10.13/lib/python3.10/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


## 1. Index ảnh + đọc ground-truth (lớp foliar)

In [4]:
rows = []
for cls in CLASSES:
    cdir = BASE/cls
    if not cdir.exists():
        raise SystemExit('Khong thay ' + str(cdir) + ' — kiem tra lai cau truc Coconut Tree Disease Dataset')
    for img in sorted(cdir.iterdir()):
        if img.suffix.lower() not in IMG_EXTS:
            continue
        rows.append(dict(
            image_id = img.stem,
            cls      = cls,
            label    = CLASSES.index(cls),
            path     = str(img.relative_to(ROOT)),
            abspath  = str(img.resolve()),
        ))
df = pd.DataFrame(rows)

print('anh index:', len(df))
print('anh theo lop:', df.cls.value_counts().to_dict())
df.head(3)

anh index: 3808
anh theo lop: {'Gray Leaf Spot': 2135, 'Leaf Rot': 1673}


,image_id,cls,label,path,abspath
0,GrayLeafSpot001,Gray Leaf Spot,0,Dataset/Coconut Tree Disease Dataset/Gray Leaf...,/Users/peggy/Documents/Projects/HK2/coconut-iq...
1,GrayLeafSpot002,Gray Leaf Spot,0,Dataset/Coconut Tree Disease Dataset/Gray Leaf...,/Users/peggy/Documents/Projects/HK2/coconut-iq...
2,GrayLeafSpot003,Gray Leaf Spot,0,Dataset/Coconut Tree Disease Dataset/Gray Leaf...,/Users/peggy/Documents/Projects/HK2/coconut-iq...


## 2. Chia K-fold theo ảnh gốc bằng hash (chống rò rỉ)

Ảnh bệnh không có bản augment → mỗi ảnh là ảnh gốc của chính nó. Fold gán tất định bằng md5 (không phụ thuộc `PYTHONHASHSEED`; cùng hàm `fold_of` với `lf6_degradation.ipynb`).

In [5]:
def fold_of(orig_id, k=K, seed=SEED):
    digest = hashlib.md5((str(seed) + ':' + orig_id).encode()).hexdigest()
    return int(digest, 16) % k

df['fold'] = df.image_id.map(fold_of)

for k in range(K):
    sub = df[df.fold == k]
    present = sorted(sub.cls.unique())
    if len(present) < len(CLASSES):
        raise SystemExit('Fold ' + str(k) + ' thieu lop: ' + str(present) + ' — giam K hoac kiem tra du lieu')
    dist = sub.cls.value_counts().to_dict()
    print('fold', k, '| anh', len(sub), '| theo lop', dist)

fold 0 | anh 746 | theo lop {'Gray Leaf Spot': 412, 'Leaf Rot': 334}
fold 1 | anh 794 | theo lop {'Gray Leaf Spot': 441, 'Leaf Rot': 353}
fold 2 | anh 771 | theo lop {'Gray Leaf Spot': 444, 'Leaf Rot': 327}
fold 3 | anh 757 | theo lop {'Gray Leaf Spot': 417, 'Leaf Rot': 340}
fold 4 | anh 740 | theo lop {'Gray Leaf Spot': 421, 'Leaf Rot': 319}


## 3. Train classifier cross-fitting → dự đoán out-of-fold

Mỗi fold `k`: train trên K−1 fold còn lại, dự đoán trên fold `k`. `confidence` = xác suất softmax của lớp dự đoán. Early-stopping theo val loss trên một phần tách phân tầng từ K−1 fold train (không đụng fold `k` để khỏi rò rỉ); giữ lại trọng số tốt nhất.

Mỗi fold train xong: ghi OOF preds (append + flush) vào `oof_predictions.csv` và lưu checkpoint `runs/fold{k}.pt`. Chạy lại bỏ qua fold đã có checkpoint (resume), không train lại từ đầu.

`num_workers=0`: nạp ảnh trong tiến trình chính — tránh lỗi pickle của `spawn` trên macOS.

In [6]:
import csv
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision import models
from torchvision import transforms
from torchvision.models import MobileNet_V3_Small_Weights
from PIL import Image

NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMGSZ, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(IMGSZ + 32),
    transforms.CenterCrop(IMGSZ),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])

class FoliarDataset(Dataset):
    def __init__(self, frame, tf):
        self.paths = list(frame.abspath)
        self.labels = list(frame.label)
        self.tf = tf
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert('RGB')
        return self.tf(img), self.labels[i]

def build_model():
    weights = MobileNet_V3_Small_Weights.IMAGENET1K_V1
    model = models.mobilenet_v3_small(weights=weights)
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, len(CLASSES))
    return model.to(DEVICE)

def class_weights(frame):
    counts = []
    for c in range(len(CLASSES)):
        counts.append(int((frame.label == c).sum()))
    total = float(sum(counts))
    weights = []
    for n in counts:
        weights.append(total / (len(CLASSES) * float(n)))
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)

def split_train_val(frame):
    # tach val phan tang theo lop, tat dinh theo SEED; val chi de early-stopping
    val_index = []
    for c in range(len(CLASSES)):
        sub = frame[frame.label == c]
        n_val = max(1, int(round(len(sub) * VAL_FRAC)))
        picked = sub.sample(
            n=n_val,
            random_state=SEED,
        )
        val_index.extend(picked.index.tolist())
    is_val = frame.index.isin(val_index)
    return frame[~is_val], frame[is_val]

def make_loader(frame, tf, shuffle):
    return DataLoader(
        FoliarDataset(frame, tf),
        batch_size=BATCH,
        shuffle=shuffle,
        num_workers=0,
    )

def val_loss(model, loader, criterion):
    model.eval()
    total = 0.0
    n = 0
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            logits = model(images)
            loss = criterion(logits, labels)
            total = total + float(loss.item()) * len(labels)
            n = n + len(labels)
    return total / n

def train_fold(train_frame):
    inner_train, inner_val = split_train_val(train_frame)
    model = build_model()
    criterion = nn.CrossEntropyLoss(weight=class_weights(inner_train))
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LR,
    )
    train_loader = make_loader(inner_train, train_tf, True)
    val_eval_loader = make_loader(inner_val, eval_tf, False)
    best_val = float('inf')
    best_state = None
    since_improve = 0
    for epoch in range(EPOCHS):
        model.train()
        running = 0.0
        for images, labels in train_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            running = running + float(loss.item())
        vloss = val_loss(model, val_eval_loader, criterion)
        print('  epoch', epoch, '| train_loss', round(running / len(train_loader), 4), '| val_loss', round(vloss, 4), flush=True)
        if vloss < best_val:
            best_val = vloss
            best_state = {}
            for name, tensor in model.state_dict().items():
                best_state[name] = tensor.detach().cpu().clone()
            since_improve = 0
        else:
            since_improve = since_improve + 1
        if since_improve >= PATIENCE:
            print('  early stop @ epoch', epoch, flush=True)
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model

def predict_fold(model, val_frame):
    model.eval()
    loader = make_loader(val_frame, eval_tf, False)
    pred_chunks = []
    conf_chunks = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            logits = model(images)
            probs = torch.softmax(logits, dim=1)
            conf, pred = torch.max(probs, dim=1)
            pred_chunks.append(pred.cpu().numpy())
            conf_chunks.append(conf.cpu().numpy())
    preds = np.concatenate(pred_chunks)
    confs = np.concatenate(conf_chunks)
    return preds, confs

def append_oof(path, fold_k, val_frame, preds, confs):
    need_header = not path.exists()
    handle = path.open('a', newline='')
    writer = csv.writer(handle)
    if need_header:
        writer.writerow(['image_id', 'fold', 'pred_class', 'confidence'])
    for row, pred, conf in zip(val_frame.itertuples(), preds, confs):
        writer.writerow([row.image_id, fold_k, CLASSES[int(pred)], float(conf)])
    handle.flush()
    handle.close()

In [7]:
OOF_CSV = OUT_DIR/'oof_predictions.csv'
pred_by_image = {}
done_folds = set()
if OOF_CSV.exists():
    prev = pd.read_csv(OOF_CSV)
    for k in range(K):
        ckpt = RUN_DIR/('fold' + str(k) + '.pt')
        if ckpt.exists() and bool((prev.fold == k).any()):
            done_folds.add(k)
    prev = prev[prev.fold.isin(done_folds)]
    prev.to_csv(OOF_CSV, index=False)
    for r in prev.itertuples():
        pred_by_image[r.image_id] = (r.pred_class, float(r.confidence))
    print('resume: bo qua fold', sorted(done_folds), '|', len(pred_by_image), 'preds da co', flush=True)

for k in range(K):
    if k in done_folds:
        continue
    tr = df[df.fold != k]
    va = df[df.fold == k]
    print('fold', k, '| train', len(tr), '| val', len(va), flush=True)
    model = train_fold(tr)
    preds, confs = predict_fold(model, va)
    append_oof(OOF_CSV, k, va, preds, confs)
    for row, pred, conf in zip(va.itertuples(), preds, confs):
        pred_by_image[row.image_id] = (CLASSES[int(pred)], float(conf))
    torch.save(model.state_dict(), RUN_DIR/('fold' + str(k) + '.pt'))
    print('  luu oof + checkpoint fold', k, flush=True)
print('oof preds:', len(pred_by_image), '->', OOF_CSV, flush=True)

fold 0 | train 3062 | val 746
  epoch 0 | train_loss 0.096 | val_loss 0.0399
  epoch 1 | train_loss 0.0188 | val_loss 0.0288
  epoch 2 | train_loss 0.0155 | val_loss 0.0292
  epoch 3 | train_loss 0.0345 | val_loss 0.0301
  epoch 4 | train_loss 0.0206 | val_loss 0.0321
  epoch 5 | train_loss 0.0254 | val_loss 0.0201
  epoch 6 | train_loss 0.0205 | val_loss 0.0207
  epoch 7 | train_loss 0.0165 | val_loss 0.029
  epoch 8 | train_loss 0.0206 | val_loss 0.0263
  epoch 9 | train_loss 0.032 | val_loss 0.0351
  epoch 10 | train_loss 0.0172 | val_loss 0.0173
  epoch 11 | train_loss 0.0112 | val_loss 0.0182
  epoch 12 | train_loss 0.0121 | val_loss 0.0268
  epoch 13 | train_loss 0.0227 | val_loss 0.0189
  epoch 14 | train_loss 0.0111 | val_loss 0.0225
  epoch 15 | train_loss 0.0108 | val_loss 0.0224
  early stop @ epoch 15
  luu oof + checkpoint fold 0
fold 1 | train 3014 | val 794
  epoch 0 | train_loss 0.1052 | val_loss 0.031
  epoch 1 | train_loss 0.0315 | val_loss 0.0225
  epoch 2 | train_lo

## 4. Correctness + cổng tin cậy + abstain → phiếu LF2

$$\lambda_2(x)=\begin{cases}\varnothing & c(x)<\tau\\ 1 & \hat{y}(x)=G(x)\\ 0 & \hat{y}(x)\neq G(x)\end{cases}$$

Lọc ảnh có `conf >= CONF_TAU`; dưới ngưỡng → abstain. Phiếu `1` nếu lớp dự đoán out-of-fold khớp lớp GT (tên thư mục), ngược lại `0`.

In [8]:
def lf2_vote(gt_class, pred):
    if pred is None:
        return np.nan
    pred_class = pred[0]
    conf = pred[1]
    if conf < CONF_TAU:
        return np.nan
    if pred_class == gt_class:
        return 1
    return 0

if not pred_by_image:
    raise SystemExit('Chua co du doan — chay buoc 3 (train) truoc')

votes = []
for r in df.itertuples():
    votes.append(lf2_vote(r.cls, pred_by_image.get(r.image_id)))
df['lf2'] = votes

n_abstain = df['lf2'].isna().sum()
print('phieu 1:', int((df['lf2'] == 1).sum()))
print('phieu 0:', int((df['lf2'] == 0).sum()))
print('abstain:', int(n_abstain), '(' + format(n_abstain / len(df), '.1%') + ')')

phieu 1: 3768
phieu 0: 26
abstain: 14 (0.4%)


## 5. Ghi phiếu

In [9]:
rows = []
for r in df.itertuples():
    rows.append(make_vote(
        lf='lf2_foliar',
        image_id=r.image_id,
        task=TASK,
        vote=r.lf2,
        source=SOURCE,
        path=r.path,
    ))
write_lf_votes(VOTES_OUT, rows)

ghi: /Users/peggy/Documents/Projects/HK2/coconut-iqa/labels/votes/lf2_foliar.csv | 3794 phieu
  2_foliar_disease        : 1=3768  0=26


PosixPath('/Users/peggy/Documents/Projects/HK2/coconut-iqa/labels/votes/lf2_foliar.csv')

Kiểm định LF2 (đối chiếu gold seed, so sánh giữa các LF) nằm ở notebook so sánh riêng, chạy sau khi mọi LF hoàn tất.